# Probe Results Analysis

Use this notebook to inspect completed feature probe runs.

It helps you:
- list available probe runs
- load one run's artifacts
- inspect the final hypothesis and confidence
- review round-by-round plans and summaries
- analyze synthetic probes, real edits, and steering results
- inspect the underlying transcript and Dolma evidence used by the agent


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import JSON, Markdown, display

OUTPUT_ROOT = Path(os.environ.get("THESIS_NEURO_OUTPUT_ROOT", Path.cwd().parent / "outputs")).expanduser()
PROBE_RUNS = OUTPUT_ROOT / "probe_runs"
if not PROBE_RUNS.exists():
    raise RuntimeError(f"No probe runs found under {PROBE_RUNS}; set THESIS_NEURO_OUTPUT_ROOT")


In [ ]:
def read_json(path: Path):
    return json.loads(path.read_text())

def read_jsonl(path: Path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

def available_probe_runs(root: Path) -> pd.DataFrame:
    rows = []
    for manifest_path in sorted(root.glob("*/*/layer_*/feature_*/manifest.json")):
        manifest = read_json(manifest_path)
        rows.append(
            {
                "bundle_id": manifest.get("bundle_id"),
                "script_scope": manifest_path.parent.parent.parent.name,
                "layer": manifest.get("layer"),
                "feature_id": manifest.get("feature_id"),
                "rounds_written": manifest.get("rounds_written"),
                "tests_written": manifest.get("tests_written"),
                "steering_rows_written": manifest.get("steering_rows_written"),
                "run_root": str(manifest_path.parent),
            }
        )
    return pd.DataFrame(rows)

probe_index = available_probe_runs(PROBE_RUNS)
probe_index


In [ ]:
# Pick one run from the index above
PROBE_ROOT = PROBE_RUNS / "gemma_2_2b" / "all_scripts" / "layer_8" / "feature_302"
assert PROBE_ROOT.exists(), PROBE_ROOT
PROBE_ROOT


In [ ]:
manifest = read_json(PROBE_ROOT / "manifest.json")
report = read_json(PROBE_ROOT / "feature_probe_report.json")
evidence = read_json(PROBE_ROOT / "feature_probe_evidence.json")
rounds = read_jsonl(PROBE_ROOT / "feature_probe_rounds.jsonl")
tests = read_jsonl(PROBE_ROOT / "feature_probe_tests.jsonl")
steering = read_jsonl(PROBE_ROOT / "feature_probe_steering.jsonl")

display(JSON(manifest, expanded=True))


In [ ]:
# Final report summary
summary = {
    "bundle_id": report.get("bundle_id"),
    "layer": report.get("layer"),
    "feature_id": report.get("feature_id"),
    "feature_label": report.get("feature_label"),
    "final_hypothesis": report.get("final_hypothesis"),
    "summary": report.get("summary"),
    "confidence": report.get("confidence"),
    "uncertainty": report.get("uncertainty"),
    "round_count": report.get("round_count"),
}
display(JSON(summary, expanded=True))


In [ ]:
display(Markdown("## Evidence For"))
pd.DataFrame({"evidence_for": report.get("evidence_for", [])})


In [ ]:
display(Markdown("## Evidence Against"))
pd.DataFrame({"evidence_against": report.get("evidence_against", [])})


In [ ]:
display(Markdown("## Open Questions"))
pd.DataFrame({"remaining_open_questions": report.get("remaining_open_questions", [])})


In [ ]:
# Round-by-round view
round_rows = []
for row in rounds:
    plan = row.get("round_plan", {})
    summary = row.get("round_summary", {})
    round_rows.append(
        {
            "round": row.get("round"),
            "trigger_hypothesis": plan.get("trigger_hypothesis"),
            "anti_trigger_hypothesis": plan.get("anti_trigger_hypothesis"),
            "agent_confidence": summary.get("agent_confidence"),
            "support_score": summary.get("support_score"),
            "positive_probe_mean": summary.get("positive_probe_mean"),
            "negative_probe_mean": summary.get("negative_probe_mean"),
            "edit_delta_mean": summary.get("edit_delta_mean"),
            "best_steering_total_delta": summary.get("best_steering_total_delta"),
            "steering_ran": summary.get("steering_ran"),
            "uncertainty": summary.get("uncertainty"),
        }
    )

round_df = pd.DataFrame(round_rows)
round_df


In [ ]:
# Test rows
tests_df = pd.DataFrame(tests)
tests_df[[
    col for col in [
        "round",
        "test_kind",
        "probe_id",
        "source_id",
        "expected_effect",
        "probe_type",
        "edit_type",
        "feature_total_activation",
        "feature_peak_activation",
        "active_token_fraction",
        "comparison_delta",
        "text",
        "edited_text",
    ] if col in tests_df.columns
]]


In [ ]:
display(Markdown("## Synthetic Probes Ranked By Activation"))
synthetic_df = tests_df[tests_df["test_kind"] == "synthetic_probe"].copy()
synthetic_df.sort_values("feature_total_activation", ascending=False)[[
    col for col in [
        "probe_id",
        "expected_effect",
        "probe_type",
        "feature_total_activation",
        "feature_peak_activation",
        "active_token_fraction",
        "text",
        "reason",
    ] if col in synthetic_df.columns
]]


In [ ]:
display(Markdown("## Real Edits Ranked By Absolute Delta"))
real_edit_df = tests_df[tests_df["test_kind"] == "real_edit"].copy()
if not real_edit_df.empty:
    real_edit_df["abs_delta"] = real_edit_df["comparison_delta"].abs()
real_edit_df.sort_values("abs_delta", ascending=False)[[
    col for col in [
        "probe_id",
        "source_id",
        "expected_effect",
        "edit_type",
        "comparison_delta",
        "feature_total_activation",
        "source_text",
        "edited_text",
        "reason",
    ] if col in real_edit_df.columns
]]


In [ ]:
# Steering rows
steering_df = pd.DataFrame(steering)
steering_df[[
    col for col in [
        "round",
        "steering_strength",
        "steering_positions",
        "steering_reason",
        "target_total_delta",
        "target_peak_delta",
        "text",
    ] if col in steering_df.columns
]]


In [ ]:
display(Markdown("## Top Positive Probes From Final Report"))
pd.DataFrame(report.get("strongest_positive_probes", []))


In [ ]:
display(Markdown("## Top Negative Probes From Final Report"))
pd.DataFrame(report.get("strongest_negative_probes", []))


In [ ]:
display(Markdown("## Strongest Counterfactual Edits From Final Report"))
pd.DataFrame(report.get("strongest_counterfactual_edits", []))


In [ ]:
# Underlying source evidence used by the agent
feature_evidence = evidence.get("feature_evidence", {})
display(Markdown("## Transcript Examples"))
pd.DataFrame(feature_evidence.get("top_transcript_examples", []))


In [ ]:
display(Markdown("## Dolma Contexts"))
dolma_df = pd.DataFrame(feature_evidence.get("top_dolma_contexts", []))
dolma_df[[
    col for col in [
        "selection_reason",
        "dominant_scale",
        "feature_activation_total",
        "feature_activation_peak",
        "window_text",
    ] if col in dolma_df.columns
]]


In [ ]:
# Optional quick plots if matplotlib is available
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    if not synthetic_df.empty:
        synthetic_df.groupby("expected_effect")["feature_total_activation"].mean().sort_values().plot(kind="barh", ax=axes[0], title="Synthetic probe mean activation")
    if not real_edit_df.empty:
        real_edit_df["comparison_delta"].plot(kind="hist", bins=10, ax=axes[1], title="Real edit activation deltas")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(f"Skipping plots: {exc}")
